<a href="https://colab.research.google.com/github/batemanrichard/ABMs_in_Python_with_Mesa/blob/main/MesaTutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

#!pip uninstall mesa -y
#!pip cache purge

try:
  import mesa
except:
  !pip install mesa==1.1 --quiet
  import mesa
import numpy as np
import math
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

%matplotlib inline

print(mesa.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1.1.0


# Helper Functions

In [2]:
def get_distance(pos_1, pos_2):
  '''
  Calculate the Euclidean distance between two points

  used in trader.move()
  '''

  x1, y1 = pos_1
  x2, y2 = pos_2
  dx = x1 - x2
  dy = y1 - y2
  return math.sqrt(dx**2 + dy**2)


def flatten(list_of_lists):
  '''
  helper function for model datacollector for trade price
  collapses agent price list into one list
  '''
  return [item for sublist in list_of_lists for item in sublist]


def geometric_mean(list_of_prices):
  '''
  find the geometric mean from a list of prices
  '''
  return np.exp(np.log(list_of_prices).mean())


def get_trade(agent):
  '''
  For agent reporter in data collector

  return list of trade partners for traders and none for other agents
  '''
  if type(agent) == Trader:
    return agent.trade_partners
  else:
    return None

# Resource Classes

In [3]:
class Sugar(mesa.Agent):
  '''
  Sugar:
  - contains an amount of sugar
  - grows one amout of sugar for each turn
  '''
  def __init__(self, unique_id, model, pos, max_sugar):
    super().__init__(unique_id, model)
    self.pos = pos
    self.amount = max_sugar
    self.max_sugar = max_sugar

  def step(self):
    '''
    Sugar growth function, adds one unit of sugar each step until max amount
    '''
    self.amount = min([self.max_sugar, self.amount + 1])
    #print(self.unique_id, self.max_sugar, self.amount)



In [4]:
class Spice(mesa.Agent):
  '''
  Spice:
  - contains an amount of spice
  - grows one amount of spice for each turn
  '''
  def __init__(self, unique_id, model, pos, max_spice):
    super().__init__(unique_id, model)
    self.pos = pos
    self.amount = max_spice # this tells us how much spice at a given time-step
    self.max_spice = max_spice # This tells us the max amount of spice at a given location

  def step(self):
    '''
    Spice growth function, adds one unit of spice each step until max amount
    '''
    self.amount = min([self.max_spice, self.amount + 1])
    #print(self.unique_id, self.max_spice, self.amount)


# Trader Agent

In [5]:
class Trader(mesa.Agent):
  '''
  Trader:
  - has a metabolism for sugar and spice
  - harvests and trades sugar and spice to survive and thrive
  '''
  def __init__(self, unique_id, model, pos, moore=False, sugar = 0,
               spice = 0, metabolism_sugar = 0, metabolism_spice = 0,
               vision = 0):
    super().__init__(unique_id, model)
    self.pos = pos
    self.moore = moore
    self.sugar = sugar
    self.spice = spice
    self.metabolism_sugar = metabolism_sugar
    self.metabolism_spice = metabolism_spice
    self.vision = vision
    self.prices = []
    self.trade_partners = []


  def get_sugar(self, pos):
    '''
    used in self.get_sugar_amount()
    used in self.eat()
    '''
    this_cell = self.model.grid.get_cell_list_contents(pos)
    for agent in this_cell:
      if type(agent) is Sugar:
        return agent
    return None


  def get_sugar_amount(self, pos):
    '''
    used in self.move() as part of self.calculate_welfare()
    '''
    sugar_patch = self.get_sugar(pos)
    if sugar_patch:
      #print("Sugar ", sugar_patch.amount)
      return sugar_patch.amount
    return 0


  def get_spice(self, pos):
    '''
    used in self.get_spice_amount()
    used in self.eat()
    '''
    this_cell = self.model.grid.get_cell_list_contents(pos)
    for agent in this_cell:
      if type(agent) is Spice:
        return agent
    return None


  def get_spice_amount(self, pos):
    '''
    used in self.move() as part of self.calculate_welfare()
    '''
    spice_patch = self.get_spice(pos)
    if spice_patch:
      #print("Spice: ", spice_patch)
      return spice_patch.amount
    return 0


  def get_trader(self, pos):
    '''
    used in self.trade_with_neighbors()
    '''
    this_cell = self.model.grid.get_cell_list_contents(pos)
    for agent in this_cell:
      if isinstance(agent, Trader):
        return agent


  def is_occupied_by_other(self, pos):
    '''
    Helper function part 1 of self.move()
    '''
    if pos == self.pos:
      # agent's position is considered unoccupied as agent can stay there
      return False;
    # Get contents of each cell in neighboorhood
    this_cell = self.model.grid.get_cell_list_contents(pos)
    for a in this_cell:
      # see if occupied by another agent
      if isinstance(a , Trader):
        return True
    return False


  def calculate_welfare(self, sugar, spice):
    '''
    helper function for self.move()
    helper function for self.trade()
    '''
    # calculate total resources
    m_total = self.metabolism_sugar + self.metabolism_spice
    # Cobb-Douglas functional form
    return sugar**(self.metabolism_sugar/m_total) * spice**(self.metabolism_spice/m_total)


  def calculate_MRS(self):
    '''
    Helper function for self.trade()

    Determine what the agent needs and what they're willing to give up
    '''
    return((self.spice/self.metabolism_spice) / (self.sugar/self.metabolism_sugar))


  def is_starved(self):
    '''
    Helper function for self.maybe_die()
    '''
    return (self.sugar <= 0) or (self.spice <= 0)


  def calculate_sell_spice_amount(self, price):
    '''
    Helper function for self.maybe_sell_spice() which is called from
    self.trade()
    '''
    if price >= 1:
      sugar = 1
      spice = int(price)
    else:
      sugar = int(1/price)
      spice = 1
    return sugar, spice


  def sell_spice(self, other, sugar, spice):
    '''
    Helper function used in maybe_sell_spice()

    exchanges the sugar and spice
    '''
    self.sugar += sugar
    other.sugar -= sugar
    self.spice -= spice
    other.spice += spice

  def maybe_sell_spice(self, other, price, welfare_self, welfare_other):
    '''
    Helper function for self.trader()
    '''
    sugar_exchanged, spice_exchanged = self.calculate_sell_spice_amount(price)

    # Assess new sugar and spice amount - what if change did occur
    self_sugar = self.sugar + sugar_exchanged
    other_sugar = other.sugar - sugar_exchanged
    self_spice = self.spice - spice_exchanged
    other_spice = other.spice + spice_exchanged

    # double check to ensure agents have enough resources
    if ((self_sugar <= 0) or
        (other_sugar <= 0) or
        (self_spice <= 0) or
        (other_spice <= 0)):
      return False

    # trade criteria #1 - are both agents better off
    both_agents_better_off = (
        (welfare_self < self.calculate_welfare(self_sugar, self_spice)) and
        (welfare_other < other.calculate_welfare(other_sugar, other_spice)))

    # trade criteria #2 - is their MRS crossing
    mrs_not_crossing = self.calculate_MRS() > other.calculate_MRS()
    #print(both_agents_better_off, mrs_not_crossing)

    # if criteria are not met
    if not (both_agents_better_off and mrs_not_crossing):
      return False

    # criteria are met, trade goes through
    self.sell_spice(other, sugar_exchanged, spice_exchanged)
    return True


  def trade(self, other):
    '''
    helper function used in trade_with_neighbors()

    other is a trader agent object
    '''
    # Sanity check
    assert self.sugar > 0
    assert self.spice > 0
    assert other.sugar > 0
    assert other.spice > 0

    # calculate the Marginal Rates of Substitution
    mrs_self = self.calculate_MRS()
    mrs_other = other.calculate_MRS()

    # calculate each agents welfare
    welfare_self = self.calculate_welfare(self.sugar, self.spice)
    welfare_other = other.calculate_welfare(other.sugar, other.spice)

    # No trade it both have similar MRS
    if math.isclose(mrs_self, mrs_other):
      #print("it was close")
      return

    # Calculate price
    price = math.sqrt(mrs_self*mrs_other)
    #print(price)

    if mrs_self > mrs_other:
      # self is a sugar buyer and spice seller
      sold = self.maybe_sell_spice(other, price, welfare_self, welfare_other)
      # no trade - criteria no met
      if not sold:
        return
    else:
      # self is a spice buyer and sugar seller
      sold = other.maybe_sell_spice(self, price, welfare_other, welfare_self)
      # no trade - criteria not met
      if not sold:
        return

    # Capture data
    self.prices.append(price)
    self.trade_partners.append(other.unique_id)

    # Continue trading while criteria are met
    self.trade(other)


  ############################################################################
  #                                                                          #
  # Main Trader Functions                                                    #
  #                                                                          #
  ############################################################################


  def move(self):
    '''
    Function for trader agent to identify optimal move for each step in 4 parts
    1 - identify all possible moves
    2 - determine which move maximises welfare
    3 - find closest best option
    4 - move
    '''
    # 1. identify all possible moves
    neighbors = [i
                  for i in self.model.grid.get_neighborhood(
                      self.pos, self.moore, True, self.vision
                  ) if not self.is_occupied_by_other(i)]
    #print(self.pos, neighbors)

    # 2. determine which move maximises welfare
    welfares = [
        self.calculate_welfare(
            self.sugar + self.get_sugar_amount(pos),
            self.spice + self.get_spice_amount(pos))
        for pos in neighbors
    ]

    #print(welfares)

    # 3. find closest best option
    max_welfare = max(welfares) # find the highest welfare in welfares
    # get the index of max welfare
    candidate_indices = [i for i in range(len(welfares))
                        if math.isclose(welfares[i], max_welfare)]
    # convert index to positions of those cells
    candidates = [neighbors[i] for i in candidate_indices]
    #print(neighbors, welfares, candidate_indices, candidates)

    min_dist = min(get_distance(self.pos, pos) for pos in candidates)

    final_candidates = [pos for pos in candidates
                        if math.isclose(get_distance(self.pos, pos), min_dist, rel_tol=1e-02)]

    self.random.shuffle(final_candidates)

    # 4. move
    self.model.grid.move_agent(self, final_candidates[0])
    #print(min_dist, final_candidates)

  def eat(self):
    '''
    Function for agents to get local resources and consume sugar and spice
    '''
    # get sugar
    sugar_patch = self.get_sugar(self.pos)
    if sugar_patch:
      self.sugar += sugar_patch.amount
      sugar_patch.amount = 0
    self.sugar -= self.metabolism_sugar

    # get spice
    spice_patch = self.get_spice(self.pos)
    if spice_patch:
      self.spice += spice_patch.amount
      spice_patch.amount = 0
    self.spice -= self.metabolism_spice

    #print(self.sugar, self.spice)


  def maybe_die(self):
    '''
    Function to remove traders who have consumer all of their sugar or spice
    '''
    if self.is_starved():
      #print(self.unique_id, self.model.schedule.get_type_count(Trader))
      self.model.grid.remove_agent(self)
      self.model.schedule.remove(self)
      #print(self.unique_id, self.model.schedule.get_type_count(Trader))


  def trade_with_neighbors(self):
    '''
    Function for trader agents to decide who to trade with in three parts
    1 - identify neighbors who can trade
    2 - trade
    3 - collect data from trade
    '''
    # 1. identify neighbors who can trade
    neighbor_agents = [self.get_trader(pos) for pos in self.model.grid.get_neighborhood(
        self.pos, self.moore, False, self.vision) if self.is_occupied_by_other(pos)]
    #print(len(neighbor_agents))

    # 2. trade
    if len(neighbor_agents) == 0:
      return

    # iterate through traders in neighboring cells and trade
    for a in neighbor_agents:
      if a:
        self.trade(a)
    return

# Model Class

In [6]:
class SugarscapeG1mt(mesa.Model):
  '''
  A model class to manage Sugarscape with Traders (G1mt)
  from Growing Artificial Societies by Axtell and Epstein
  '''
  def __init__(self, width = 50, height = 50, initial_population = 200,
               endowment_min = 25, endowment_max = 50, metabolism_min = 1,
               metabolism_max = 5, vision_min = 1, vision_max = 5):

    # Initiate width and height of sugarscape
    self.width = width
    self.height = height

    # Initiate population attributes
    self.initial_population = initial_population
    self.endowment_min = endowment_min
    self.endowment_max = endowment_max
    self.metabolism_min = metabolism_min
    self.metabolism_max = metabolism_max
    self.vision_min = vision_min
    self.vision_max = vision_max
    self.running = True

    # Initiate mesa grid class
    self.grid = mesa.space.MultiGrid(self.width, self.height, torus = False)

    # Initiate scheduler
    self.schedule = mesa.time.RandomActivationByType(self)

    # Initiate datacollector
    self.datacollector = mesa.DataCollector(
        model_reporters = {"Trader": lambda m: m.schedule.get_type_count(Trader),
          "Trade Volume": lambda m: sum(len(a.trade_partners)
          for a in m.schedule.agents_by_type[Trader].values()),
          "Price": lambda m: geometric_mean(flatten([a.prices for a in m.schedule.agents_by_type[Trader].values()]))},
        #agent_reporters = {"Trade Network": lambda a: get_trade(a)}
    )

    # Read in landscape file from supplementary material
    sugar_distribution = np.genfromtxt("/content/drive/MyDrive/Colab Notebooks/sugar-map.txt")
    spice_distribution = np.flip(sugar_distribution, 1)

    # For loop to create each agent
    agent_id = 0
    for _, x, y in self.grid.coord_iter():

      # Sugar Agents
      max_sugar = sugar_distribution[x,y]
      if max_sugar > 0:
        sugar = Sugar(agent_id, self, (x,y), max_sugar)
        self.grid.place_agent(sugar, (x,y))
        self.schedule.add(sugar)
        #print(self.schedule.agents_by_type[Sugar][agent_id])
        agent_id += 1

    for _, x, y in self.grid.coord_iter():
      max_spice = spice_distribution[x,y]
      if max_spice > 0:
        spice = Spice(agent_id, self, (x,y), max_spice)
        self.grid.place_agent(spice, (x,y))
        self.schedule.add(spice)
        #print(self.schedule.agents_by_type[Spice][agent_id])
        agent_id += 1

    for i in range(self.initial_population):
      # get agent position
      x = self.random.randrange(self.width)
      y = self.random.randrange(self.height)
      # Give agents initial endowment
      sugar = int(self.random.uniform(self.endowment_min, self.endowment_max+1))
      spice = int(self.random.uniform(self.endowment_min, self.endowment_max+1))
      # Give agents initial metabolism
      metabolism_sugar = int(self.random.uniform(self.metabolism_min, self.metabolism_max+1))
      metabolism_spice = int(self.random.uniform(self.metabolism_min, self.metabolism_max+1))
      # Give agents initial vision
      vision = int(self.random.uniform(self.vision_min, self.vision_max+1))
      # Create trader object
      trader = Trader(agent_id,
                      self,
                      (x,y),
                      moore = False,
                      sugar = sugar,
                      spice = spice,
                      metabolism_sugar = metabolism_sugar,
                      metabolism_spice = metabolism_spice,
                      vision = vision)
      # Place agent
      self.grid.place_agent(trader, (x,y))
      self.schedule.add(trader)
      #print(trader.unique_id, trader.pos, trader.sugar, trader.metabolism_spice)
      agent_id +=1


  def randomise_traders(self):
    '''
    Helper function for self.step()

    Creates a randomised list of traders to remove impact of first- or
    second-mover advantage
    '''
    trader_shuffle = list(self.schedule.agents_by_type[Trader].values())
    self.random.shuffle(trader_shuffle)
    return trader_shuffle


  def step(self):
    '''
    Unique step function that does staged activation of sugar and spice
    and then randomly activates traders
    '''

    # Step sugar agents
    for sugar in self.schedule.agents_by_type[Sugar].values():
      sugar.step()

    # Step spice agents
    for spice in self.schedule.agents_by_type[Spice].values():
      spice.step()

    # Step trader agents
    # to account for agent death and removal we need a seperate data structure to
    # iterate
    trader_shuffle = self.randomise_traders()
    for agent in trader_shuffle:
      agent.prices = []
      agent.trade_partners = []
      agent.move()
      agent.eat()
      agent.maybe_die()

    trader_shuffle = self.randomise_traders()
    for agent in trader_shuffle:
      agent.trade_with_neighbors()
      #print(agent.prices, agent.trade_partners)

    # Move the steps forward one
    self.schedule.steps += 1 #important for data collector to track number of days

    # collect model level data
    self.datacollector.collect(self)

  def run_model(self, step_count=1000):

    for i in range (step_count):
      #print(i)
      self.step()


# Run Sugarscape

In [7]:
model = SugarscapeG1mt()
model.run_model()

/tmp/ipykernel_135416/2606161587.py:27: RuntimeWarning: Mean of empty slice.
  return np.exp(np.log(list_of_prices).mean())
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


# Analyse Data

In [8]:
results = model.datacollector.get_model_vars_dataframe()

results

,Trader,Trade Volume,Price
0,200,567,1.132408
1,200,179,1.124082
2,200,131,1.095671
3,200,115,1.054518
4,199,100,1.140294
...,...,...,...
995,55,6,1.002176
996,55,6,1.046593
997,55,2,1.195553
998,55,7,1.014764


In [9]:
# retrieve agent level results
agent_results = model.datacollector.get_agent_vars_dataframe()
# filter out Nones from non-trader agents
agent_results = agent_results[agent_results["Trade Network"].notnull()]

agent_results

KeyError: 'Trade Network'

In [ ]:
# Plot number of agents per time step
results.plot(y = "Trader", use_index = True)

In [ ]:
# Plot trade price per step
y = list(results["Price"])
x = range(1000)

plt.scatter(x, y , s = 1)

In [ ]:
# Plot trade volume

plt.bar(results.index, results["Trade Volume"])

In [ ]:
# PLot trade volume improved

for i in range(1000):
  plt.vlines(i,0,results["Trade Volume"][i])

In [ ]:
# create graph object
#G = nx.Graph()

# add agent keys to make initial node set
#G.add_nodes_from(list(model.schedule.agents_by_type[Trader].keys()))

# create edge list
#for idx, row in agent_results.iterrows():
#  if len(row["Trade Network"]) > 0:
#    for agent in row["Trade Network"]:
#      G.add_edge(idx[1], agent)

In [ ]:
#nx.node_connectivity(G), nx.average_clustering(G), nx.diameter(G), nx.global_efficiency(G)

In [ ]:
#degree = [d for n,d in G.degree()]
#plt.hist(degree)

In [ ]:
#nx.draw(G)

# Batch Run and Analysis

In [ ]:
params = {"width": 50, "height": 50,
          "vision_min": range(1, 3),
          "metabolism_max": [3, 5]}

results_batch = mesa.batch_run(
    SugarscapeG1mt,
    parameters = params,
    iterations = 1,
    number_processes = 1,
    data_collection_period = 1,
    display_progress = True
)

In [ ]:
results_df = pd.DataFrame(results_batch)
results_df

In [ ]:
# Does price still converge to 1
plt.scatter(results_df["Step"], results_df["Price"], s = 0.75)

In [ ]:
results_explore = results_df[results_df["metabolism_max"]== 5]
results_explore

In [ ]:
plt.scatter(results_explore["Step"], results_explore["Price"], s = 0.75)

In [ ]:
for i in range(4):
  results_explore = results_df[results_df["RunId"] == i]
  plt.plot(results_explore["Step"], results_explore["Trader"])